# 🚒 🚓 🚗 Navintrix AI — Unified 7-Class YOLO11 Training Pipeline (Production Ready)

**Purpose**: Train a unified YOLO11 model on **both** standard traffic (UA-DETRAC) and emergency vehicles (Roboflow Emergency dataset), strictly adhering to the 7-class contract required by `configs/model.yaml` and `detection/detector.py`.

---

### The 7-Class Contract

| Class ID | Class Name | Category | Source Dataset | System Role |
|:---:|---|---|---|---|
| **0** | `car` | Normal | UA-DETRAC (`ved-k3yao`) | Traffic volume, density, adaptive signal timing |
| **1** | `motorcycle` | Normal | UA-DETRAC / Overlap | Vehicle counting, queue estimation |
| **2** | `bus` | Normal | UA-DETRAC (`ved-k3yao`) | Transit prioritization |
| **3** | `truck` | Normal | UA-DETRAC (`van` $\to$ `truck`) | Heavy vehicle PCU weighting |
| **4** | `ambulance` | **Emergency** | Roboflow (`emergency-vehicles-xug80`) | Emergency preemption & Green Wave (`emergency/priority.py`) |
| **5** | `fire_truck` | **Emergency** | Roboflow (`emergency-vehicles-xug80`) | Emergency preemption & Green Wave (`emergency/priority.py`) |
| **6** | `police_vehicle` | **Emergency** | Roboflow (`emergency-vehicles-xug80`) | Emergency preemption & Green Wave (`emergency/priority.py`) |

---

### ⚙️ Required Kaggle Settings (Right Sidebar)
1. **Accelerator**: Settings $\to$ Accelerator $\to$ **GPU T4 x2** (or GPU P100)
2. **Internet**: Settings $\to$ Internet $\to$ **On**


---
## 1. Pre-Flight Hardware & Environment Checkpoint

Checks for GPU availability and installs required packages.


In [ ]:
import sys
import os
import torch

# CHECKPOINT 1: Hardware verification
print("=" * 60)
print("🔍 PRE-FLIGHT CHECKPOINT: HARDWARE & ACCELERATOR")
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU DETECTED: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"   CUDA Version : {torch.version.cuda}")
    print(f"   Device Count : {torch.cuda.device_count()}")
else:
    print("❌ ERROR: No GPU detected!")
    print("   Training on CPU will be extremely slow.")
    print("   👉 In Kaggle right sidebar: Settings -> Accelerator -> select 'GPU T4 x2'")
    raise SystemError("GPU is required for training YOLO11. Enable GPU in notebook settings.")

!pip install -q ultralytics roboflow pyyaml pillow matplotlib tqdm

import ultralytics
import roboflow
print(f"\n✅ Ultralytics version: {ultralytics.__version__}")
print(f"✅ Roboflow version   : {roboflow.__version__}")


---
## 2. Download Datasets from Roboflow with Validation Checkpoint

Downloads both pre-annotated datasets directly into Kaggle.


In [ ]:
from pathlib import Path
from roboflow import Roboflow

ROBOFLOW_API_KEY = "rUaYPil0HTiiws78ompp"
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# 1. Download UA-DETRAC (forked by you)
print("📥 1/2 Downloading UA-DETRAC from Roboflow...")
try:
    project_detrac = rf.workspace("ved-k3yao").project("ua-detrac-riedy-mn8ry")
    dataset_detrac = project_detrac.version(1).download("yolov11")
    DETRAC_RAW_DIR = Path(dataset_detrac.location)
    print(f"✅ UA-DETRAC downloaded to: {DETRAC_RAW_DIR}")
except Exception as e:
    print(f"❌ Failed to download UA-DETRAC: {e}")
    raise

# 2. Download Emergency Vehicles
print("\n📥 2/2 Downloading Emergency Vehicles from Roboflow...")
try:
    project_emer = rf.workspace("luis-lheslie-judilla-lmona").project("emergency-vehicles-xug80")
    dataset_emer = project_emer.version(12).download("yolov11")
    EMERGENCY_RAW_DIR = Path(dataset_emer.location)
    print(f"✅ Emergency Vehicles downloaded to: {EMERGENCY_RAW_DIR}")
except Exception as e:
    print(f"❌ Failed to download Emergency Vehicles: {e}")
    raise

# CHECKPOINT 2: Verify downloaded contents
detrac_imgs = list(DETRAC_RAW_DIR.rglob("*.jpg")) + list(DETRAC_RAW_DIR.rglob("*.png"))
emer_imgs = list(EMERGENCY_RAW_DIR.rglob("*.jpg")) + list(EMERGENCY_RAW_DIR.rglob("*.png"))

print("\n" + "=" * 60)
print("📊 DOWNLOAD VERIFICATION CHECKPOINT")
print("=" * 60)
print(f"  UA-DETRAC Images Found : {len(detrac_imgs):,}")
print(f"  Emergency Images Found : {len(emer_imgs):,}")

assert len(detrac_imgs) > 0, f"No images found in UA-DETRAC download ({DETRAC_RAW_DIR})"
assert len(emer_imgs) > 0, f"No images found in Emergency download ({EMERGENCY_RAW_DIR})"
print("✅ Both datasets verified on disk.")


---
## 3. Canonical 7-Class Configuration & Synonym Mapping


In [ ]:
import yaml
import shutil
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# CANONICAL 7 CLASSES from configs/model.yaml
ALL_CLASSES = [
    "car",              # 0
    "motorcycle",       # 1
    "bus",              # 2
    "truck",            # 3
    "ambulance",        # 4
    "fire_truck",       # 5
    "police_vehicle",   # 6
]
CLASS_TO_ID = {name: i for i, name in enumerate(ALL_CLASSES)}
NUM_CLASSES = len(ALL_CLASSES)

WORK_DIR      = Path("/kaggle/working")
UNIFIED_DIR   = WORK_DIR / "data" / "unified_7class"
TRAIN_DIR     = WORK_DIR / "runs"

# Hyperparameters
YOLO_VARIANT  = "yolo11n.pt"
EPOCHS        = 50
IMG_SIZE      = 640
BATCH_SIZE    = 16
SAVE_PERIOD   = 5   # Saves a model checkpoint every 5 epochs (epoch5.pt, epoch10.pt...)
PATIENCE      = 15  # Early stopping patience if validation metrics plateau

def normalize_label_str(val) -> str:
    s = str(val).lower().strip()
    return s.replace("-", "_").replace(" ", "_")

# UA-DETRAC Mapping
DETRAC_MAPPING = {
    "car": "car",
    "bus": "bus",
    "van": "truck",
    "truck": "truck",
    "motorcycle": "motorcycle",
    "motorbike": "motorcycle",
    "others": None,
    "0": "car",
    "1": "motorcycle",
    "2": "bus",
    "3": "truck",
}

# Emergency Mapping
EMERGENCY_MAPPING = {
    # Ambulance
    "ambulance": "ambulance",
    "amb": "ambulance",
    "emergency_ambulance": "ambulance",
    
    # Fire truck / engine
    "fire_truck": "fire_truck",
    "firetruck": "fire_truck",
    "fire_engine": "fire_truck",
    
    # Police
    "police_vehicle": "police_vehicle",
    "police_car": "police_vehicle",
    "police": "police_vehicle",
    "cop": "police_vehicle",
    
    # Overlapping civilian vehicles in emergency set
    "car": "car",
    "bus": "bus",
    "truck": "truck",
    "motorcycle": "motorcycle",
}

print("✅ Canonical 7-Class configuration initialized.")
print(f"Target classes (nc={NUM_CLASSES}): {ALL_CLASSES}")


---
## 4. Merge Both Datasets into Unified 7-Class Format with Quality Checks


In [ ]:
def load_dataset_names(ds_dir: Path) -> list[str]:
    for yml in list(ds_dir.glob("*.yaml")) + list(ds_dir.glob("*.yml")):
        try:
            with open(yml, encoding="utf-8") as f:
                d = yaml.safe_load(f) or {}
            names = d.get("names", [])
            if isinstance(names, list):
                return [normalize_label_str(n) for n in names]
            elif isinstance(names, dict):
                return [normalize_label_str(names[k]) for k in sorted(names)]
        except Exception:
            pass
    return []

def merge_yolo_datasets():
    splits = ["train", "valid", "val", "test"]
    for s in ["train", "val", "test"]:
        (UNIFIED_DIR / "images" / s).mkdir(parents=True, exist_ok=True)
        (UNIFIED_DIR / "labels" / s).mkdir(parents=True, exist_ok=True)

    summary = {
        "detrac_images": 0,
        "emergency_images": 0,
        "class_counts": {c: 0 for c in ALL_CLASSES},
        "dropped_boxes": 0,
        "invalid_boxes": 0,
    }

    # 1. Process UA-DETRAC
    detrac_names = load_dataset_names(DETRAC_RAW_DIR)
    print(f"UA-DETRAC parsed names: {detrac_names}")

    for sp in splits:
        target_sp = "val" if sp in ("valid", "val") else sp
        img_src_dir = DETRAC_RAW_DIR / sp / "images"
        lbl_src_dir = DETRAC_RAW_DIR / sp / "labels"
        if not img_src_dir.is_dir():
            continue

        for img_p in img_src_dir.glob("*"):
            if img_p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
                continue
            lbl_p = lbl_src_dir / f"{img_p.stem}.txt"
            if not lbl_p.is_file():
                continue

            yolo_lines = []
            for line in lbl_p.read_text(encoding="utf-8", errors="replace").splitlines():
                tokens = line.strip().split()
                if len(tokens) < 5:
                    summary["invalid_boxes"] += 1
                    continue
                try:
                    raw_cls_id = int(float(tokens[0]))
                    cx, cy, bw, bh = (float(v) for v in tokens[1:5])
                except ValueError:
                    summary["invalid_boxes"] += 1
                    continue

                # Bounds check
                if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0 and 0.0 < bw <= 1.0 and 0.0 < bh <= 1.0):
                    summary["invalid_boxes"] += 1
                    continue

                src_name = detrac_names[raw_cls_id] if 0 <= raw_cls_id < len(detrac_names) else str(raw_cls_id)
                target_cls = DETRAC_MAPPING.get(src_name) or DETRAC_MAPPING.get(normalize_label_str(src_name))
                if target_cls is None:
                    summary["dropped_boxes"] += 1
                    continue

                new_id = CLASS_TO_ID[target_cls]
                yolo_lines.append(f"{new_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                summary["class_counts"][target_cls] += 1

            dst_stem = f"detrac_{img_p.stem}"
            shutil.copy2(img_p, UNIFIED_DIR / "images" / target_sp / f"{dst_stem}{img_p.suffix}")
            (UNIFIED_DIR / "labels" / target_sp / f"{dst_stem}.txt").write_text("\n".join(yolo_lines), encoding="utf-8")
            summary["detrac_images"] += 1

    # 2. Process Emergency Vehicles
    emer_names = load_dataset_names(EMERGENCY_RAW_DIR)
    print(f"Emergency dataset parsed names: {emer_names}")

    for sp in splits:
        target_sp = "val" if sp in ("valid", "val") else sp
        img_src_dir = EMERGENCY_RAW_DIR / sp / "images"
        lbl_src_dir = EMERGENCY_RAW_DIR / sp / "labels"
        if not img_src_dir.is_dir():
            continue

        for img_p in img_src_dir.glob("*"):
            if img_p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
                continue
            lbl_p = lbl_src_dir / f"{img_p.stem}.txt"
            if not lbl_p.is_file():
                continue

            yolo_lines = []
            for line in lbl_p.read_text(encoding="utf-8", errors="replace").splitlines():
                tokens = line.strip().split()
                if len(tokens) < 5:
                    summary["invalid_boxes"] += 1
                    continue
                try:
                    raw_cls_id = int(float(tokens[0]))
                    cx, cy, bw, bh = (float(v) for v in tokens[1:5])
                except ValueError:
                    summary["invalid_boxes"] += 1
                    continue

                if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0 and 0.0 < bw <= 1.0 and 0.0 < bh <= 1.0):
                    summary["invalid_boxes"] += 1
                    continue

                src_name = emer_names[raw_cls_id] if 0 <= raw_cls_id < len(emer_names) else str(raw_cls_id)
                norm_name = normalize_label_str(src_name)
                target_cls = EMERGENCY_MAPPING.get(norm_name)
                if target_cls is None:
                    if "amb" in norm_name:
                        target_cls = "ambulance"
                    elif "fire" in norm_name:
                        target_cls = "fire_truck"
                    elif "polic" in norm_name or "cop" in norm_name:
                        target_cls = "police_vehicle"

                if target_cls is None or target_cls not in CLASS_TO_ID:
                    summary["dropped_boxes"] += 1
                    continue

                new_id = CLASS_TO_ID[target_cls]
                yolo_lines.append(f"{new_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                summary["class_counts"][target_cls] += 1

            dst_stem = f"emer_{img_p.stem}"
            shutil.copy2(img_p, UNIFIED_DIR / "images" / target_sp / f"{dst_stem}{img_p.suffix}")
            (UNIFIED_DIR / "labels" / target_sp / f"{dst_stem}.txt").write_text("\n".join(yolo_lines), encoding="utf-8")
            summary["emergency_images"] += 1

    # Safety check: if val is empty, copy 15% from train
    val_count = len(list((UNIFIED_DIR / "images" / "val").glob("*")))
    if val_count == 0:
        print("⚠️ Warning: Validation set was empty. Creating val split from train images...")
        train_imgs = sorted((UNIFIED_DIR / "images" / "train").glob("*"))
        n_val = max(10, int(len(train_imgs) * 0.15))
        for img_p in train_imgs[:n_val]:
            shutil.move(img_p, UNIFIED_DIR / "images" / "val" / img_p.name)
            lbl_p = UNIFIED_DIR / "labels" / "train" / f"{img_p.stem}.txt"
            if lbl_p.exists():
                shutil.move(lbl_p, UNIFIED_DIR / "labels" / "val" / lbl_p.name)

    # Write unified data.yaml
    unified_yaml = {
        "path": str(UNIFIED_DIR),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test" if list((UNIFIED_DIR / "images" / "test").glob("*")) else "images/val",
        "nc": NUM_CLASSES,
        "names": {i: name for i, name in enumerate(ALL_CLASSES)},
    }
    with open(UNIFIED_DIR / "data.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(unified_yaml, f, sort_keys=False)

    return summary

summary = merge_yolo_datasets()

# CHECKPOINT 3: Validate merge integrity
print("\n" + "=" * 60)
print("🔍 MERGE VALIDATION CHECKPOINT")
print("=" * 60)
print(f"  UA-DETRAC Images : {summary['detrac_images']:,}")
print(f"  Emergency Images : {summary['emergency_images']:,}")
print(f"  Total Images     : {summary['detrac_images'] + summary['emergency_images']:,}")
print(f"  Valid Boxes      : {sum(summary['class_counts'].values()):,}")
print(f"  Invalid Boxes    : {summary['invalid_boxes']}")
print(f"  Dropped Boxes    : {summary['dropped_boxes']:,}")

train_count = len(list((UNIFIED_DIR / "images" / "train").glob("*")))
val_count = len(list((UNIFIED_DIR / "images" / "val").glob("*")))
print(f"  Train Image Count: {train_count:,}")
print(f"  Val Image Count  : {val_count:,}")

assert train_count > 0, "Train split has 0 images! Merging failed."
assert val_count > 0, "Validation split has 0 images! Training cannot evaluate."

print("\nClass Bounding Box Breakdown:")
for c in ALL_CLASSES:
    cat = "EMERGENCY" if c in ["ambulance", "fire_truck", "police_vehicle"] else "NORMAL"
    print(f"  [{CLASS_TO_ID[c]}] {c:16s} ({cat:9s}) : {summary['class_counts'][c]:,} boxes")
print("✅ Merge checkpoint PASSED.")


### 4.1 Class Distribution Visualization


In [ ]:
classes = ALL_CLASSES
counts = [summary["class_counts"][c] for c in classes]
colors = ["#3498db", "#e67e22", "#2ecc71", "#9b59b6", "#f1c40f", "#e74c3c", "#1abc9c"]

plt.figure(figsize=(12, 5))
bars = plt.bar(classes, counts, color=colors)
plt.title("Bounding Boxes per Class (Unified 7-Class Dataset)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Class Name", fontsize=12)
plt.ylabel("Annotation Count", fontsize=12)
plt.xticks(rotation=15, fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(max(counts)*0.015, 10),
             f"{count:,}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(WORK_DIR / "class_distribution_7classes.png", dpi=150)
plt.show()


### 4.2 Visual Verification Spot-Check


In [ ]:
def spot_check(n=6):
    img_dir = UNIFIED_DIR / "images" / "train"
    lbl_dir = UNIFIED_DIR / "labels" / "train"
    imgs = sorted(img_dir.glob("*"))
    if not imgs:
        return

    em_imgs = [p for p in imgs if p.name.startswith("emer_")]
    dt_imgs = [p for p in imgs if p.name.startswith("detrac_")]

    selected = []
    if em_imgs:
        selected.extend(random.sample(em_imgs, min(3, len(em_imgs))))
    if dt_imgs:
        selected.extend(random.sample(dt_imgs, min(n - len(selected), len(dt_imgs))))
    if not selected:
        selected = random.sample(imgs, min(n, len(imgs)))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, img_p in zip(axes.flat, selected):
        im = Image.open(img_p).convert("RGB")
        draw = ImageDraw.Draw(im)
        iw, ih = im.size

        lbl_p = lbl_dir / f"{img_p.stem}.txt"
        if lbl_p.exists():
            for line in lbl_p.read_text(encoding="utf-8").splitlines():
                toks = line.strip().split()
                if len(toks) == 5:
                    cid = int(toks[0])
                    cx, cy, bw, bh = (float(v) for v in toks[1:])
                    x1 = (cx - bw/2) * iw
                    y1 = (cy - bh/2) * ih
                    x2 = (cx + bw/2) * iw
                    y2 = (cy + bh/2) * ih
                    col = colors[cid] if cid < len(colors) else "#ffffff"
                    draw.rectangle([x1, y1, x2, y2], outline=col, width=3)
                    draw.text((x1 + 3, max(0, y1 - 12)), ALL_CLASSES[cid], fill=col)

        ax.imshow(im)
        ax.set_title(img_p.name[:25], fontsize=9)
        ax.axis("off")

    plt.suptitle("Sanity Check: Ground Truth Bounding Boxes Across Normal & Emergency Vehicles", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

spot_check()


---
## 5. Fine-Tune YOLO11 with Automated Checkpointing

**Checkpointing Configuration**:
- `save=True`: Saves `best.pt` and `last.pt` on every epoch.
- `save_period=5`: Saves intermediate epoch checkpoints (`epoch5.pt`, `epoch10.pt`, etc.) so work is never lost if a session drops.
- `patience=15`: Early stopping to prevent overfitting if metrics plateau.
- `workers=2`: Safe multiprocessing on Kaggle cloud instances.

> 💡 **Resume Tip**: If training is ever interrupted, you can resume by running:
> ```python
> model = YOLO('/kaggle/working/runs/yolo11_7class_unified/weights/last.pt')
> model.train(resume=True)
> ```


In [ ]:
from ultralytics import YOLO

data_yaml_path = UNIFIED_DIR / "data.yaml"

print("=" * 60)
print(f"🚀 LAUNCHING YOLO11 7-CLASS TRAINING")
print("=" * 60)
print(f"Model       : {YOLO_VARIANT}")
print(f"Dataset     : {data_yaml_path}")
print(f"Epochs      : {EPOCHS}")
print(f"Batch Size  : {BATCH_SIZE}")
print(f"Save Period : Every {SAVE_PERIOD} epochs")

model = YOLO(YOLO_VARIANT)

results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(TRAIN_DIR),
    name="yolo11_7class_unified",
    exist_ok=True,
    # Checkpointing & Early Stopping
    save=True,
    save_period=SAVE_PERIOD,
    patience=PATIENCE,
    # Optimization
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    workers=2,
    # Augmentations for illumination & angle resilience
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    seed=42,
    verbose=True,
    plots=True,
)


---
## 6. Evaluate Validation Metrics Across All 7 Classes


In [ ]:
metrics = model.val()

print("\n" + "=" * 60)
print("📊 OVERALL VALIDATION METRICS")
print("=" * 60)
print(f"  mAP50         : {metrics.box.map50:.4f}")
print(f"  mAP50-95      : {metrics.box.map:.4f}")
print(f"  Mean Precision: {metrics.box.mp:.4f}")
print(f"  Mean Recall   : {metrics.box.mr:.4f}")

print("\n" + "=" * 60)
print("🎯 PER-CLASS AP50 BREAKDOWN (ALL 7 CLASSES)")
print("=" * 60)
if hasattr(metrics.box, "ap_class_index") and metrics.box.ap_class_index is not None:
    for i, cls_idx in enumerate(metrics.box.ap_class_index):
        cls_name = ALL_CLASSES[cls_idx] if cls_idx < len(ALL_CLASSES) else f"cls_{cls_idx}"
        cat = "EMERGENCY" if cls_name in ["ambulance", "fire_truck", "police_vehicle"] else "NORMAL"
        ap50 = metrics.box.ap50[i]
        print(f"  [{cls_idx}] {cls_name:16s} ({cat:9s}) : AP50 = {ap50:.4f}")


### 6.1 Training Curves & Confusion Matrix


In [ ]:
run_dir = TRAIN_DIR / "yolo11_7class_unified"
plot_files = ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]

for p_name in plot_files:
    p_path = run_dir / p_name
    if p_path.exists():
        im = Image.open(p_path)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(im)
        ax.set_title(p_name, fontsize=12, fontweight="bold")
        ax.axis("off")
        plt.tight_layout()
        plt.show()


---
## 7. Model Export & Smoke Test Checkpoint

Validates model weights and exports `best.pt` to `/kaggle/working/`.


In [ ]:
best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

print("=" * 60)
print("🔍 POST-TRAINING SMOKE TEST & EXPORT CHECKPOINT")
print("=" * 60)

assert best_pt.exists(), f"ERROR: best.pt was not found at {best_pt}!"
size_mb = best_pt.stat().st_size / (1024 * 1024)
assert size_mb > 2.0, f"ERROR: best.pt is unexpectedly small ({size_mb:.2f} MB)!"

# Copy weights to root for easy 1-click download from Kaggle UI
export_best = WORK_DIR / "best.pt"
shutil.copy2(best_pt, export_best)
print(f"✅ best.pt successfully verified ({size_mb:.1f} MB)")
print(f"   Exported to: {export_best}")

if last_pt.exists():
    shutil.copy2(last_pt, WORK_DIR / "last.pt")
    print(f"   (last.pt also exported to {WORK_DIR / 'last.pt'})")

# Smoke test inference to verify weights load and predict correctly
smoke_test_model = YOLO(str(export_best))
val_imgs = sorted((UNIFIED_DIR / "images" / "val").glob("*"))
if val_imgs:
    test_pred = smoke_test_model.predict(str(val_imgs[0]), conf=0.25, verbose=False)
    print(f"✅ SMOKE TEST PASSED: Successfully loaded {export_best.name} and ran inference!")
    print(f"   Detected {len(test_pred[0].boxes)} objects in test image {val_imgs[0].name}")

print("\n" + "=" * 60)
print("📥 DOWNLOAD INSTRUCTION:")
print("=" * 60)
print("1. In Kaggle file explorer (right side), locate '/kaggle/working/best.pt'")
print("2. Click the three dots (...) -> Download")
print("3. Place the file in your local repository at: traffic-ai/models/best.pt")


### 7.1 Visual Predictions on Validation Images


In [ ]:
test_imgs = sorted((UNIFIED_DIR / "images" / "val").glob("*"))[:6]
if test_imgs and best_pt.exists():
    trained_model = YOLO(str(export_best))
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, img_p in zip(axes.flat, test_imgs):
        res = trained_model.predict(str(img_p), conf=0.35, verbose=False)
        annotated = res[0].plot()
        ax.imshow(annotated[:, :, ::-1])
        ax.set_title(f"{img_p.name[:25]} ({len(res[0].boxes)} dets)", fontsize=9)
        ax.axis("off")
    plt.suptitle("Model Predictions on Validation Samples (7-Class Model)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(WORK_DIR / "inference_samples_7class.png", dpi=150)
    plt.show()


---
## 8. Save Metrics & Summary Record


In [ ]:
import json

training_record = {
    "pipeline": "unified_7class_roboflow",
    "classes": ALL_CLASSES,
    "model": YOLO_VARIANT,
    "epochs": EPOCHS,
    "dataset_summary": summary,
    "metrics": {
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    },
    "exported_weights": "best.pt",
}

summary_file = WORK_DIR / "training_summary_7class.json"
summary_file.write_text(json.dumps(training_record, indent=2), encoding="utf-8")
print(f"📄 Summary record saved to: {summary_file}")
print(json.dumps(training_record, indent=2))
